In [1]:
import os
import pandas as pd
import numpy as np

# Anchor to project root explicitly — works regardless of launch location
os.chdir('/Users/eldigardonaz/Documents/WorldCup-2026-Analytics')
print("Working directory:", os.getcwd())

results = pd.read_csv('data/raw/historical/results.csv')
elo = pd.read_csv('data/raw/wc2026/elo_ratings_wc2026.csv')
matches_2026 = pd.read_csv('data/raw/wc2026/matches_detailed.csv')

print("Loaded successfully ✓")
print("Historical:", len(results), "rows")

Working directory: /Users/eldigardonaz/Documents/WorldCup-2026-Analytics
Loaded successfully ✓
Historical: 49477 rows


In [2]:
# What team names exist in each dataset?
hist_teams = set(results['home_team'].unique()) | set(results['away_team'].unique())

print("Historical dataset team count:", len(hist_teams))
print("\nElo file columns:", elo.columns.tolist())
print("\n2026 file columns:", matches_2026.columns.tolist())

Historical dataset team count: 336

Elo file columns: ['year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']

2026 file columns: ['match_id', 'date', 'kickoff_time_utc', 'stage_name', 'stadium_name', 'city', 'country', 'home_team_name', 'home_fifa_code', 'away_team_name', 'away_fifa_code', 'home_score', 'away_score', 'status', 'home_xg', 'away_xg', 'home_goalkeeper', 'away_goalkeeper', 'player_of_the_match_name', 'referee_name']


In [3]:
# Get the 48 teams in the 2026 World Cup (from Elo, which has is_host and codes)
wc_2026_teams = elo[elo['year'] == 2026][['country', 'country_code']].drop_duplicates()
print("2026 World Cup teams with their FIFA codes:")
print(wc_2026_teams.to_string(index=False))
print("\nTotal:", len(wc_2026_teams))

2026 World Cup teams with their FIFA codes:
               country country_code
                 Spain           ES
             Argentina           AR
                France           FR
               England           EN
                Brazil           BR
              Portugal           PT
              Colombia           CO
           Netherlands           NL
               Ecuador           EC
               Croatia           HR
               Germany           DE
                Norway           NO
                 Japan           JP
                Turkey           TR
               Uruguay           UY
           Switzerland           CH
               Senegal           SN
               Belgium           BE
                Mexico           MX
              Paraguay           PY
               Austria           AT
               Morocco           MA
                Canada           CA
             Australia           AU
              Scotland           SQ
                  Ir

In [4]:
# The 48 WC team names as they appear in the Elo file
elo_wc_names = set(elo[elo['year'] == 2026]['country'].unique())

# All team names in historical data
hist_teams = set(results['home_team'].unique()) | set(results['away_team'].unique())

# Which WC teams from Elo do NOT have an exact match in historical data?
missing = elo_wc_names - hist_teams
print("WC teams whose Elo name doesn't match historical data:")
for team in sorted(missing):
    print("  ", team)
print(f"\n{len(missing)} of 48 teams need name mapping")

WC teams whose Elo name doesn't match historical data:
   Czechia

1 of 48 teams need name mapping


In [5]:
# Confirm what the historical data calls Czechia/Czech Republic
czech_variants = [t for t in hist_teams if 'zech' in t]
print("Czech-related names in historical data:", czech_variants)

Czech-related names in historical data: ['Czech Republic', 'Czechoslovakia']


In [6]:
# Team name standardisation — map Elo/2026 names to historical dataset convention
TEAM_NAME_MAP = {
    'Czechia': 'Czech Republic',
}

# Apply to the Elo dataframe
elo['country'] = elo['country'].replace(TEAM_NAME_MAP)

# Verify the fix worked
elo_wc_names_fixed = set(elo[elo['year'] == 2026]['country'].unique())
still_missing = elo_wc_names_fixed - hist_teams
print("Remaining mismatches after fix:", still_missing)
print("Should be empty set() →", len(still_missing), "remaining")

Remaining mismatches after fix: set()
Should be empty set() → 0 remaining


In [7]:
# Look at how Elo ratings are stored across time for one team
brazil_elo = elo[elo['country'] == 'Brazil'][['year', 'snapshot_date', 'rating', 'rank']].sort_values('year')
print("Brazil's Elo across years:")
print(brazil_elo.tail(10).to_string(index=False))

print("\nElo year range:", elo['year'].min(), "→", elo['year'].max())
print("Snapshots per team per year (Brazil sample):")
print(elo[elo['country'] == 'Brazil'].groupby('year').size().tail(5))

Brazil's Elo across years:
 year snapshot_date  rating  rank
 2018    2018-12-31    2136     1
 2019    2019-12-31    2081     2
 2020    2020-12-31    2135     1
 2021    2021-12-31    2149     1
 2022    2022-12-31    2133     2
 2023    2023-12-31    2011     6
 2024    2024-12-31    1996     5
 2025    2025-12-31    1978     6
 2026    2026-12-31    1984     5
 2026    2026-05-27    1984     5

Elo year range: 1901 → 2026
Snapshots per team per year (Brazil sample):
year
2022    1
2023    1
2024    1
2025    1
2026    2
dtype: int64


In [8]:
# --- Build a clean Elo lookup: one rating per team per year ---
# For 2026, keep only the pre-tournament snapshot (May 27), drop the year-end projection
elo_clean = elo[~((elo['year'] == 2026) & (elo['snapshot_date'] == '2026-12-31'))].copy()

# Keep just what we need: team, year, rating
elo_lookup = elo_clean[['country', 'year', 'rating']].drop_duplicates(subset=['country', 'year'])

# --- Prepare historical results ---
results['date'] = pd.to_datetime(results['date'])
results['match_year'] = results['date'].dt.year

# Anti-leakage: a match in year Y uses the rating from year Y-1
results['elo_year'] = results['match_year'] - 1

# --- Join home team Elo ---
results = results.merge(
    elo_lookup.rename(columns={'country': 'home_team', 'year': 'elo_year', 'rating': 'home_elo'}),
    on=['home_team', 'elo_year'],
    how='left'
)

# --- Join away team Elo ---
results = results.merge(
    elo_lookup.rename(columns={'country': 'away_team', 'year': 'elo_year', 'rating': 'away_elo'}),
    on=['away_team', 'elo_year'],
    how='left'
)

# --- The feature itself ---
results['elo_diff'] = results['home_elo'] - results['away_elo']

# --- Check how many matches got Elo ratings ---
print("Total matches:", len(results))
print("Matches with both Elo ratings:", results['elo_diff'].notna().sum())
print("Matches missing Elo (old/minor teams):", results['elo_diff'].isna().sum())

Total matches: 49477
Matches with both Elo ratings: 7522
Matches missing Elo (old/minor teams): 41955


In [9]:
# Does elo_diff actually predict match outcomes?
# Filter to matches that have Elo data and a result
tested = results[results['elo_diff'].notna()].copy()

# Recreate the result column for this subset
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

tested['result'] = tested.apply(get_result, axis=1)

# Average elo_diff for each outcome
print("Average Elo difference by match result:")
print(tested.groupby('result')['elo_diff'].mean().round(1))
print("\nWhat this should show:")
print("- Home Win → positive (home team was stronger)")
print("- Away Win → negative (away team was stronger)")
print("- Draw → near zero (teams evenly matched)")

Average Elo difference by match result:
result
Away Win   -98.8
Draw       -10.7
Home Win    79.8
Name: elo_diff, dtype: float64

What this should show:
- Home Win → positive (home team was stronger)
- Away Win → negative (away team was stronger)
- Draw → near zero (teams evenly matched)


In [10]:
# --- Rolling form: build a long-format "team match log" first ---
# Each match becomes TWO rows: one from home team's perspective, one from away's.
# This lets us track each team's form regardless of home/away.

# Home perspective
home_log = results[results['elo_diff'].notna()][['date', 'home_team', 'home_score', 'away_score']].copy()
home_log.columns = ['date', 'team', 'goals_for', 'goals_against']
home_log['points'] = home_log.apply(
    lambda r: 3 if r['goals_for'] > r['goals_against'] else (1 if r['goals_for'] == r['goals_against'] else 0),
    axis=1
)

# Away perspective
away_log = results[results['elo_diff'].notna()][['date', 'away_team', 'away_score', 'home_score']].copy()
away_log.columns = ['date', 'team', 'goals_for', 'goals_against']
away_log['points'] = away_log.apply(
    lambda r: 3 if r['goals_for'] > r['goals_against'] else (1 if r['goals_for'] == r['goals_against'] else 0),
    axis=1
)

# Combine and sort chronologically
team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)

print("Team match log built:", len(team_log), "team-match rows")
print(team_log.head())

Team match log built: 15044 team-match rows
        date       team  goals_for  goals_against  points
0 1902-05-03    England        2.0            2.0       1
1 1902-05-03   Scotland        2.0            2.0       1
2 1903-04-04    England        1.0            2.0       0
3 1903-04-04   Scotland        2.0            1.0       3
4 1903-09-13  Argentina        2.0            3.0       0


In [11]:
# --- REBUILD team_log from scratch (previous cell corrupted it) ---
base = results[results['elo_diff'].notna()]

# Home perspective
home_log = base[['date', 'home_team', 'home_score', 'away_score']].copy()
home_log.columns = ['date', 'team', 'goals_for', 'goals_against']

# Away perspective
away_log = base[['date', 'away_team', 'away_score', 'home_score']].copy()
away_log.columns = ['date', 'team', 'goals_for', 'goals_against']

# Points from each team's perspective
for log in (home_log, away_log):
    log['points'] = np.where(log['goals_for'] > log['goals_against'], 3,
                     np.where(log['goals_for'] == log['goals_against'], 1, 0))

# Combine + sort by team then date
team_log = pd.concat([home_log, away_log]).sort_values(['team', 'date']).reset_index(drop=True)

# --- Rolling form with leakage protection via transform ---
def rolling_prior_mean(s, window=5):
    return s.shift(1).rolling(window, min_periods=1).mean()

team_log['form_points'] = team_log.groupby('team')['points'].transform(rolling_prior_mean)
team_log['form_goals_for'] = team_log.groupby('team')['goals_for'].transform(rolling_prior_mean)
team_log['form_goals_against'] = team_log.groupby('team')['goals_against'].transform(rolling_prior_mean)

# --- Verify ---
print("team_log rebuilt:", len(team_log), "rows")
print("Columns:", team_log.columns.tolist())
print("\nArgentina's earliest matches:")
arg = team_log[team_log['team'] == 'Argentina'][
    ['date', 'goals_for', 'goals_against', 'points', 'form_points']
].head(8)
print(arg.to_string(index=False))
print("\nFirst row form_points is NaN →", pd.isna(arg.iloc[0]['form_points']))

team_log rebuilt: 15044 rows
Columns: ['date', 'team', 'goals_for', 'goals_against', 'points', 'form_points', 'form_goals_for', 'form_goals_against']

Argentina's earliest matches:
      date  goals_for  goals_against  points  form_points
1903-09-13        2.0            3.0       0          NaN
1905-08-15        0.0            0.0       1     0.000000
1906-08-15        2.0            0.0       3     0.500000
1906-10-21        2.0            1.0       3     1.333333
1908-08-15        2.0            2.0       1     1.750000
1908-09-13        2.0            1.0       3     1.600000
1908-10-04        0.0            1.0       0     2.200000
1909-08-15        2.0            1.0       3     2.000000

First row form_points is NaN → True


In [12]:
# --- Host nation flag (complete history 1930–2026) ---
WC_HOSTS = {
    1930: ['Uruguay'],
    1934: ['Italy'],
    1938: ['France'],
    1950: ['Brazil'],
    1954: ['Switzerland'],
    1958: ['Sweden'],
    1962: ['Chile'],
    1966: ['England'],
    1970: ['Mexico'],
    1974: ['Germany'],          # West Germany (Test to make sure)
    1978: ['Argentina'],
    1982: ['Spain'],
    1986: ['Mexico'],
    1990: ['Italy'],
    1994: ['United States'],
    1998: ['France'],
    2002: ['South Korea', 'Japan'],
    2006: ['Germany'],
    2010: ['South Africa'],
    2014: ['Brazil'],
    2018: ['Russia'],
    2022: ['Qatar'],
    2026: ['United States', 'Canada', 'Mexico'],
}

def is_host_match(row):
    hosts = WC_HOSTS.get(row['match_year'], [])
    home_host = 1 if row['home_team'] in hosts else 0
    away_host = 1 if row['away_team'] in hosts else 0
    return pd.Series([home_host, away_host])

results['home_is_host'] = 0
results['away_is_host'] = 0

wc_mask = results['tournament'] == 'FIFA World Cup'
results.loc[wc_mask, ['home_is_host', 'away_is_host']] = results[wc_mask].apply(is_host_match, axis=1).values

# --- Verify across multiple eras ---
print("Host matches found per World Cup year:")
host_matches = results[(results['home_is_host']==1) | (results['away_is_host']==1)]
print(host_matches.groupby('match_year').size())

Host matches found per World Cup year:
match_year
1930     4
1934     5
1938     2
1950     6
1954     4
1958     6
1962     6
1966     6
1970     4
1974     7
1978     7
1982     5
1986     5
1990     7
1994     4
1998     7
2002    11
2006     7
2010     3
2014     7
2018     5
2022     3
2026     9
dtype: int64


In [13]:
# What stages exist in the 2026 data?
print("2026 stage names:")
print(matches_2026['stage_name'].value_counts())

2026 stage names:
stage_name
Group Stage    72
Name: count, dtype: int64


In [14]:
# --- Stage weighting: is_knockout flag ---
# 2026 live data: read directly from stage_name
# Historical: approximate — knockout matches are the elimination rounds

# For the historical results, we don't have a clean stage column.
# Simple, honest approximation: flag based on known knockout stage names if present,
# otherwise default group stage (0). Most predictive signal comes from Elo + form anyway.

KNOCKOUT_KEYWORDS = ['final', 'semi', 'quarter', 'round of', 'knockout', 'third place']

def infer_knockout(tournament_or_stage):
    text = str(tournament_or_stage).lower()
    return 1 if any(kw in text for kw in KNOCKOUT_KEYWORDS) else 0

# Apply to historical (tournament column rarely has stage, so most → 0, which is fine)
results['is_knockout'] = results['tournament'].apply(infer_knockout)

# --- Report ---
print("Historical knockout matches flagged:", results['is_knockout'].sum())
print("\nNote: 2026 World Cup is still in group stage —")
print("no knockout matches exist yet. This feature activates")
print("as the tournament progresses into Round of 32+.")
print("\nFor 2026 specifically, stage comes from the live 'stage_name' column.")

Historical knockout matches flagged: 1

Note: 2026 World Cup is still in group stage —
no knockout matches exist yet. This feature activates
as the tournament progresses into Round of 32+.

For 2026 specifically, stage comes from the live 'stage_name' column.


In [15]:
# --- ASSEMBLE FINAL MODEL-READY DATASET ---
import os

# 1. Start from matches that have Elo, add the target (what we're predicting)
def encode_result(row):
    if row['home_score'] > row['away_score']:
        return 'Home Win'
    elif row['home_score'] < row['away_score']:
        return 'Away Win'
    else:
        return 'Draw'

model_df = results[results['elo_diff'].notna()].copy()
model_df['result'] = model_df.apply(encode_result, axis=1)
rows_before = len(model_df)

# 2. Form lookup from team_log (dedupe (date, team) to prevent merge fan-out)
form_lookup = team_log[['date', 'team', 'form_points', 'form_goals_for', 'form_goals_against']
                       ].drop_duplicates(subset=['date', 'team'])

# 3. Attach HOME team form
model_df = model_df.merge(
    form_lookup.rename(columns={'team': 'home_team',
        'form_points': 'home_form_pts', 'form_goals_for': 'home_form_gf', 'form_goals_against': 'home_form_ga'}),
    on=['date', 'home_team'], how='left')

# 4. Attach AWAY team form
model_df = model_df.merge(
    form_lookup.rename(columns={'team': 'away_team',
        'form_points': 'away_form_pts', 'form_goals_for': 'away_form_gf', 'form_goals_against': 'away_form_ga'}),
    on=['date', 'away_team'], how='left')

# 5. Derived difference features
model_df['form_pts_diff'] = model_df['home_form_pts'] - model_df['away_form_pts']
model_df['host_diff'] = model_df['home_is_host'] - model_df['away_is_host']

# --- CRITICAL CHECK: did the merge duplicate any rows? ---
print("Rows before merge:", rows_before)
print("Rows after merge: ", len(model_df))
print("MATCH (should be equal):", rows_before == len(model_df))
print("\nMissing form (first-ever matches per team):", model_df['home_form_pts'].isna().sum())

Rows before merge: 7522
Rows after merge:  7522
MATCH (should be equal): True

Missing form (first-ever matches per team): 21


In [16]:
# --- Drop unusable rows (missing form) and save ---
final_df = model_df.dropna(subset=['home_form_pts', 'away_form_pts']).copy()

# Select the columns the model actually needs
feature_cols = [
    'date', 'home_team', 'away_team', 'tournament', 'match_year',
    'elo_diff', 'home_elo', 'away_elo',
    'home_form_pts', 'away_form_pts', 'form_pts_diff',
    'home_form_gf', 'away_form_gf', 'home_form_ga', 'away_form_ga',
    'home_is_host', 'away_is_host', 'host_diff', 'is_knockout',
    'result'
]
final_df = final_df[feature_cols]

# Save to processed folder
import os
os.makedirs('../data/processed', exist_ok=True)
final_df.to_csv('../data/processed/model_ready.csv', index=False)

print("Final model-ready dataset saved ✓")
print("Rows:", len(final_df))
print("Features:", len(feature_cols) - 4, "(excluding date/teams/target)")
print("\nResult distribution:")
print(final_df['result'].value_counts())
print("\nSaved to: ../data/processed/model_ready.csv")

Final model-ready dataset saved ✓
Rows: 7482
Features: 16 (excluding date/teams/target)

Result distribution:
result
Home Win    3460
Away Win    2101
Draw        1921
Name: count, dtype: int64

Saved to: ../data/processed/model_ready.csv
